# 09 — YOLO Leaderboard Improvement · serious training + holdout + submission

**Objective:** move beyond the 6-epoch / frame-step-6 baselines. This notebook fine-tunes YOLO11s on **more training frames** (step 3), for **18 epochs**, at **960 px**, with carefully tracked validation and inference settings. It also provides an optional YOLO11m comparison.

- The *existing* `01_EDA_YOLO_Baseline` and `05_YOLO_Tuning` experiments are **not modified**.
- Model selection uses the **original 7-video validation holdout**; no validation video enters training. This holdout must not be called a blind/unbiased test after repeated tuning.
- **Kaggle leaderboard 0.22163** is a *different* evaluation set. **Do not compare it directly** to CV/holdout COCO mAP. Only a Kaggle submission can determine whether we beat it.
- Kaggle `/kaggle/working` is session-local unless the output is committed/downloaded. Checkpoints are saved after each epoch.
- Training stages are **manual**, not all launched by `Run All`. Do not run another training cell while one is active.

Training may exceed Kaggle time/GPU allocations depending on accelerator. A smoke test, explicit completion checks, and resumable checkpoints are included. The entire training/inference has **not** been executed in this environment.

## 1 — Install and verify hardware / competition input

In [1]:
%pip -q install ultralytics pycocotools pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from collections import defaultdict
import os, json, math, time, gc, random, shutil, zipfile
import numpy as np
import pandas as pd
import yaml
import torch
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

SEED = 42
CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
CANDIDATES = [
    Path('/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026'),
    Path('/kaggle/input/multiple-object-video-detection-hw-2-ucu-ai-2026'),
]
DATA = next((p for p in CANDIDATES if (p/'train/annotations_coco.json').exists()), None)
assert DATA is not None, 'Attach the competition dataset using Kaggle Add Input.'
assert torch.cuda.is_available(), 'Enable GPU in Kaggle notebook Settings.'
OUT = Path('/kaggle/working/yolo_leaderboard_v1')
SPLITS, RUNS, REPORTS = (OUT/x for x in ['splits', 'runs', 'reports'])
for p in (SPLITS, RUNS, REPORTS): p.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Dataset:', DATA)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU RAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1e9,2))
print('Working disk GB free:', round(shutil.disk_usage('/kaggle/working').free/1e9,1))
import ultralytics
print('Ultralytics version:', ultralytics.__version__)
print('Dataset folders:', [p.name for p in DATA.iterdir()])

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Dataset: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
GPU: Tesla T4
GPU RAM GB: 15.64
Working disk GB free: 20.9
Ultralytics version: 8.4.204
Dataset folders: ['val', 'test_images.csv', 'test', 'train']


## 2 — Separate training videos from original validation videos

We use all **train videos** (sampled every third frame), not the fold-0 training subset. The original `val/` sequences provide an **out-of-video holdout**. We do **not** add original validation frames to training at this stage. This creates a stronger model by using more data while keeping a separate assessment set.

YOLO label files must be present under `train/labels/` and `val/labels/`. COCO GT and class order are checked independently.

In [3]:
TRAIN_JSON = DATA/'train/annotations_coco.json'
VAL_JSON = DATA/'val/annotations_coco.json'
assert TRAIN_JSON.is_file() and VAL_JSON.is_file()
with TRAIN_JSON.open() as f: train_gt = json.load(f)
with VAL_JSON.open() as f: val_gt = json.load(f)

def as_table(d):
    rows = []
    for im in d['images']:
        fn = im['file_name']
        rows.append({'image_id': im['id'], 'file_name': fn,
                     'clip': Path(fn).parent.name, 'width': im['width'], 'height': im['height']})
    return pd.DataFrame(rows).sort_values(['clip','file_name']).reset_index(drop=True)

def sample_by_clip(d, step):
    return (d.groupby('clip', group_keys=False)
             .apply(lambda x: x.iloc[::step], include_groups=False)
             .reset_index(drop=True).assign(clip=lambda x: x['file_name'].map(lambda s: Path(s).parent.name)))

train_meta, holdout_meta = as_table(train_gt), as_table(val_gt)
TRAIN_STEP = 3          # Prior baseline was 6; 3 uses ~twice as many train frames.
HOLDOUT_STEP = 3        # Faster holdout metric, but remains VIDEO-disjoint.
train_selected = sample_by_clip(train_meta, TRAIN_STEP)
val_selected = sample_by_clip(holdout_meta, HOLDOUT_STEP)
assert set(train_meta['clip']).isdisjoint(set(holdout_meta['clip'])), 'Train / holdout video leakage!'
assert set(int(c['id']) for c in train_gt['categories']) == set(range(1,9))
assert set(int(c['id']) for c in val_gt['categories']) == set(range(1,9))
assert len(train_selected)>5000 and len(val_selected)>400
print('Train images:', len(train_meta), '=> sampled:',len(train_selected), '| clips:',train_meta['clip'].nunique())
print('Holdout images:',len(holdout_meta),' => sampled:',len(val_selected),'| clips:',holdout_meta['clip'].nunique())
print('Train clip overlap with holdout:',len(set(train_meta['clip']) & set(holdout_meta['clip'])))
train_selected.to_csv(SPLITS/'train_step3.csv',index=False)
val_selected.to_csv(SPLITS/'holdout_step3.csv',index=False)

Train images: 24201 => sampled: 8090 | clips: 56
Holdout images: 2846  => sampled: 951 | clips: 7
Train clip overlap with holdout: 0


In [4]:
def image_path(split, file_name):
    return DATA/split/file_name

def label_path(image):
    # Original format: split/images/<clip>/<frame>.jpg => split/labels/<clip>/<frame>.txt
    s = str(image)
    assert '/images/' in s, f'Unexpected image path: {s}'
    return Path(s.replace('/images/','/labels/')).with_suffix('.txt')

paths = {'train': [image_path('train',x) for x in train_selected['file_name']],
         'val':   [image_path('val',x) for x in val_selected['file_name']]}
for split, pp in paths.items():
    missing_img = [str(p) for p in pp if not p.is_file()]
    missing_lbl = [str(p) for p in pp if not label_path(p).is_file()]
    assert not missing_img, f'{split}: images missing: {missing_img[:3]}'
    assert not missing_lbl, f'{split}: labels missing: {missing_lbl[:3]}'
    (SPLITS/f'{split}.txt').write_text(chr(10).join(map(str, pp)) + chr(10))
    print(split, len(pp), 'images and labels verified')
DATA_YAML = SPLITS/'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'train':str(SPLITS/'train.txt'),
    'val':str(SPLITS/'val.txt'),
    'nc':8,
    'names':dict(enumerate(CLASSES)),
},sort_keys=False))
print('YAML:', DATA_YAML)

train 8090 images and labels verified
val 951 images and labels verified
YAML: /kaggle/working/yolo_leaderboard_v1/splits/data.yaml


## 3 — Choose initialization and configure primary experiment

**Default:** train the full sampled training set starting from **official `yolo11s.pt`**. This gives a clean longer-run experiment, independent from the previous fold-0 checkpoint. If you consciously want to continue from your 6-epoch tuned fold-0 model, change `INIT_WEIGHTS` to the checkpoint path — this is a different warm-start experiment and must be labeled as such.

We change data quantity, epoch budget and augmentation together; therefore improvements cannot be attributed to any *single* change. Earlier fold-0 tuning already tested LR, optimizer, and resolution independently.

In [5]:
PRIMARY_NAME = 'yolo11s_960_step3_18ep_adamw'
INIT_WEIGHTS = 'yolo11s.pt'  # Official COCO pretrained weights, not a prior fold checkpoint.
PRIMARY = dict(
    imgsz=960,
    epochs=18,
    batch=2,
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.1,
    warmup_epochs=2.0,
    weight_decay=0.0005,
    cos_lr=True,
    mosaic=0.25,
    close_mosaic=5,
    mixup=0.0,
    degrees=0.0,
    translate=0.10,
    scale=0.30,
    fliplr=0.5,
    patience=18,
)
print('Primary experiment:',PRIMARY_NAME)
print('Start weights:',INIT_WEIGHTS)
print('Settings:',PRIMARY)

Primary experiment: yolo11s_960_step3_18ep_adamw
Start weights: yolo11s.pt
Settings: {'imgsz': 960, 'epochs': 18, 'batch': 2, 'optimizer': 'AdamW', 'lr0': 0.0005, 'lrf': 0.1, 'warmup_epochs': 2.0, 'weight_decay': 0.0005, 'cos_lr': True, 'mosaic': 0.25, 'close_mosaic': 5, 'mixup': 0.0, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.3, 'fliplr': 0.5, 'patience': 18}


## 4 — Training with checkpoint / resume guards

`best.pt`, `last.pt`, and `results.csv` are written to `/kaggle/working/yolo_leaderboard_v1/runs/<run>`. `epoch*.pt` are not kept to avoid filling disk. Re-running the cell **skips a complete run**, or **resumes** an interrupted run when `last.pt` exists. Training checkpoints remain temporary until you export or commit them. Keep original notebook code available to recreate the YAML when resuming in a new Kaggle session.

In [6]:
def train_run(name, weights, cfg):
    run = RUNS/name
    best, last = run/'weights/best.pt', run/'weights/last.pt'
    history, marker = run/'results.csv',run/'completed.json'
    if marker.exists() and best.is_file():
        print('Already complete, skipping:',name)
        return best
    if history.exists() and best.is_file() and len(pd.read_csv(history)) >= cfg['epochs']:
        marker.write_text(json.dumps({'name':name,'epochs':len(pd.read_csv(history)), 'best':str(best)},indent=2))
        print('Verified complete from history, skipping:',name)
        return best

    t0=time.monotonic()
    if last.is_file():
        print('Resuming interrupted run:',last)
        model=YOLO(str(last))
        model.train(resume=True,device=0,workers=2)
    else:
        print('Starting new training:',name)
        model=YOLO(str(weights))
        model.train(
            data=str(DATA_YAML), device=0, workers=2, cache=False,
            seed=SEED, deterministic=True, amp=True,
            save=True, save_period=-1, plots=False,
            project=str(RUNS),name=name,exist_ok=True,
            **cfg
        )
    assert best.exists(),f'Missing best.pt: {best}'
    assert history.exists(),f'Missing results.csv: {history}'
    n_epochs=len(pd.read_csv(history))
    assert n_epochs>=cfg['epochs'], f'Only {n_epochs}/{cfg["epochs"]} epochs; not marking complete.'
    marker.write_text(json.dumps({'name':name,'epochs':n_epochs,
                                  'init_weights':str(weights),'settings':cfg,
                                  'minutes_this_session':round((time.monotonic()-t0)/60,2)},indent=2))
    print('TRAINING COMPLETE:',name,'epochs:',n_epochs)
    del model
    gc.collect();torch.cuda.empty_cache()
    return best

### 4A — Run primary training (GPU-intensive)

**Run this cell manually.** Do not use `Run All` automatically when you are not ready for a long GPU experiment. If it OOMs, do not delete existing `last.pt`; set a smaller image size/batch **only for a new run** (changing settings mid-resume invalidates a controlled comparison).

In [8]:
primary_best = train_run(PRIMARY_NAME, INIT_WEIGHTS, PRIMARY)
print('Primary best checkpoint:', primary_best)

Starting new training: yolo11s_960_step3_18ep_adamw
Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/yolo_leaderboard_v1/splits/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=18, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.25, multi_sc

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/18      1.85G      1.542     0.9585      1.027        209        960: 100% ━━━━━━━━━━━━ 4045/4045 12.8it/s 5:170.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 238/238 13.1it/s 18.2s0.1s
                   all        951      38206       0.55      0.405       0.41      0.214

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/18      1.85G      1.492     0.8859      1.012        106        960: 100% ━━━━━━━━━━━━ 4045/4045 12.8it/s 5:160.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 238/238 13.3it/s 17.9s0.1s
                   all        951      38206      0.447      0.407      0.365      0.183

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/18      1.96G       1.45     0.8336     0.9984         81        960: 100% ━━━━━━━━━━━━ 4045/4045 12.8it/s 5:170.1ss
                

## 5 — Holdout COCOeval with original `iscrowd` and exact image IDs

Same category mapping (YOLO 0..7 → COCO 1..8), **COCOeval maxDets=100**, original holdout ground truth and same box format as our Faster R-CNN/RetinaNet evaluation. We collect predictions at `conf=0.001`; lower than typical deployment thresholds because COCOeval ranks detections by score. Do not use Ultralytics validation mAP as a substitute for this common evaluator.

In [9]:
HOLDOUT_COCO=COCO(str(VAL_JSON))
assert set(int(c['id']) for c in HOLDOUT_COCO.dataset['categories'])==set(range(1,9))

def evaluate_holdout(checkpoint, tag, imgsz=960, conf=0.001, iou=0.7, max_det=300, batch=4):
    checkpoint=Path(checkpoint)
    assert checkpoint.exists(),checkpoint
    folder=REPORTS/tag;folder.mkdir(parents=True,exist_ok=True)
    metrics_path=folder/'holdout_coco_metrics.json'
    preds_path=folder/'holdout_predictions_coco.json'
    if metrics_path.exists() and preds_path.exists():
        print('Evaluation cached:',metrics_path)
        return json.loads(metrics_path.read_text())
    model=YOLO(str(checkpoint))
    paths=[str(image_path('val',s)) for s in val_selected['file_name']]
    ids=[int(x) for x in val_selected['image_id']]
    preds=[]
    begin=time.monotonic()
    for i in range(0,len(paths),batch):
        rr=model.predict(source=paths[i:i+batch],imgsz=imgsz,device=0,
                         batch=batch,conf=conf,iou=iou,max_det=max_det,
                         augment=False,verbose=False,save=False)
        assert len(rr)==len(paths[i:i+batch])
        for im_id,r in zip(ids[i:i+batch],rr):
            if r.boxes is None: continue
            for box, cls, score in zip(r.boxes.xyxy.cpu().numpy(),
                                       r.boxes.cls.cpu().numpy(),
                                       r.boxes.conf.cpu().numpy()):
                x1,y1,x2,y2=map(float,box)
                w,h=max(0.,x2-x1),max(0.,y2-y1)
                if w<=0 or h<=0:continue
                assert 0<=int(cls)<8
                preds.append({'image_id':im_id,'category_id':int(cls)+1,
                              'bbox':[x1,y1,w,h],'score':float(score)})
        if (i//batch+1)%50==0:print('Evaluated images:',min(i+batch,len(paths)),'/',len(paths))
    assert len(preds)>0,'No detections. Verify checkpoint/confidence.'
    preds_path.write_text(json.dumps(preds))
    coco_dt=HOLDOUT_COCO.loadRes(preds)
    ev=COCOeval(HOLDOUT_COCO,coco_dt,'bbox')
    ev.params.imgIds=ids
    ev.params.catIds=list(range(1,9))
    ev.params.maxDets=[1,10,100]
    ev.evaluate();ev.accumulate();ev.summarize()
    ss=ev.stats
    result={
        'tag':tag,'checkpoint':str(checkpoint),
        'holdout_images':len(ids),'detections':len(preds),
        'mAP50-95':float(ss[0]),'mAP50':float(ss[1]),'mAP75':float(ss[2]),
        'AP_small':float(ss[3]),'AP_medium':float(ss[4]),'AP_large':float(ss[5]),
        'inference_ms_per_image':round((time.monotonic()-begin)*1000/len(ids),1),
        'imgsz':imgsz,'conf':conf,'iou':iou,'max_det_prediction':max_det,
        'coco_max_dets':100,'uses_original_iscrowd':True,
    }
    metrics_path.write_text(json.dumps(result,indent=2))
    print('Saved holdout metrics:',metrics_path)
    del model,ev,coco_dt
    gc.collect();torch.cuda.empty_cache()
    return result

loading annotations into memory...
Done (t=0.45s)
creating index...
index created!


### 5A — Evaluate primary model on *original* holdout (no training)

In [10]:
primary_best=RUNS/PRIMARY_NAME/'weights/best.pt'
assert primary_best.is_file(), 'Finish 4A training first.'
primary_metrics=evaluate_holdout(primary_best,PRIMARY_NAME,imgsz=PRIMARY['imgsz'])
display(pd.DataFrame([primary_metrics])[['tag','mAP50-95','mAP50','AP_small','AP_medium','AP_large','inference_ms_per_image']])

Evaluated images: 200 / 951
Evaluated images: 400 / 951
Evaluated images: 600 / 951
Evaluated images: 800 / 951
Loading and preparing results...
DONE (t=1.42s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=47.95s).
Accumulating evaluation results...
DONE (t=1.66s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.184
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.350
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.166
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.074
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.162
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.318
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.084
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.213
 Average Recall     (AR) @[ IoU=0.5

,tag,mAP50-95,mAP50,AP_small,AP_medium,AP_large,inference_ms_per_image
0,yolo11s_960_step3_18ep_adamw,0.184291,0.350345,0.073577,0.162045,0.317645,115.5


## 7 — Compare with attached historical models on this **same holdout** (optional)

The previous fold 0 weights were trained on only a subset of *train videos* and can be scored on the original holdout. However, their training set size and epoch budget differ — this is an end-to-end comparison, **not an isolated hyperparameter effect**. Attach the previous YOLO notebook output via **Add Input** first.

In [11]:
old_weights=sorted(Path('/kaggle/input').rglob('yolo_experiments/yolo11s_960_adamw/weights/best.pt'))
print('Found previous YOLO11s 960 checkpoints:',old_weights)
if old_weights:
    print('OPTIONAL: uncomment to evaluate old 6-epoch model on the same original holdout')
    # old_metrics=evaluate_holdout(old_weights[0],'historical_yolo11s_960_6ep_holdout',imgsz=960)
else:
    print('Attach old baseline as Kaggle Input if you want a same-holdout comparison.')

Found previous YOLO11s 960 checkpoints: [PosixPath('/kaggle/input/notebooks/margopaduchak/01-eda-yolo-baseline/yolo_experiments/yolo11s_960_adamw/weights/best.pt')]
OPTIONAL: uncomment to evaluate old 6-epoch model on the same original holdout


## 8 — Competition submission (optional, after assessing holdout)

**Format is grounded in the earlier successfully created competition CSV:** `image_id,PredictionString`; each detection is `class_id confidence x_min y_min width height`. Class IDs are **0-based** for competition output. We require a one-to-one image-ID mapping, numerical predictions, and no duplicates.

This uses *all* test images (no temporal sampling). Inference settings `conf=0.01, iou=0.65, max_det=300` are reasonable starting points, not guaranteed optimal. Do not repeatedly tune on public leaderboard; first evaluate on the holdout. **A valid CSV does not guarantee a leaderboard score higher than 0.22163.**

In [12]:
# Discover competition template if provided; otherwise use our established filename convention.
TEMPLATE_CANDIDATES=sorted(DATA.rglob('*sample*submission*.csv'))
print('Sample submission candidates:',TEMPLATE_CANDIDATES[:8])
TEST_IMAGES=sorted((DATA/'test/images').rglob('*.jpg')) + sorted((DATA/'test/images').rglob('*.png'))
TEST_IMAGES=sorted(set(TEST_IMAGES))
assert TEST_IMAGES, 'No test images found under test/images; inspect competition dataset layout.'

def test_image_id(path):
    # Previously produced CSV used e.g. clip_001_0000001
    return f'{path.parent.name}_{path.stem}'

if TEMPLATE_CANDIDATES:
    template=pd.read_csv(TEMPLATE_CANDIDATES[0])
    assert {'image_id','PredictionString'}.issubset(template.columns)
    wanted_ids=template['image_id'].astype(str).tolist()
else:
    template=None
    wanted_ids=[test_image_id(p) for p in TEST_IMAGES]

image_dict={test_image_id(p):p for p in TEST_IMAGES}
assert len(image_dict)==len(TEST_IMAGES), 'Duplicate test image IDs!'
assert len(wanted_ids)==len(set(wanted_ids)), 'Duplicate template IDs!'
assert set(wanted_ids)==set(image_dict), (
    f'Template/test mismatch: missing={list(set(wanted_ids)-set(image_dict))[:5]}, '
    f'extra={list(set(image_dict)-set(wanted_ids))[:5]}'
)
print('Verified test images:',len(TEST_IMAGES))
print('Sample IDs:',wanted_ids[:5])
print('Test IDs mapping verified.')

Sample submission candidates: []
Verified test images: 3811
Sample IDs: ['clip_001_0000001', 'clip_001_0000002', 'clip_001_0000003', 'clip_001_0000004', 'clip_001_0000005']
Test IDs mapping verified.


In [13]:
# CHOOSE checkpoint to submit. Default = stronger primary model.
SUBMIT_CHECKPOINT=RUNS/PRIMARY_NAME/'weights/best.pt'
SUBMIT_IMGSZ=PRIMARY['imgsz']
SUBMIT_CONF=0.01
SUBMIT_IOU=0.65
SUBMIT_MAX_DET=300
assert SUBMIT_CHECKPOINT.is_file(), 'Train the primary model before inference.'
print('Submission checkpoint:',SUBMIT_CHECKPOINT)
print('Do not compare holdout mAP directly with Kaggle baseline 0.22163.')

Submission checkpoint: /kaggle/working/yolo_leaderboard_v1/runs/yolo11s_960_step3_18ep_adamw/weights/best.pt
Do not compare holdout mAP directly with Kaggle baseline 0.22163.


### 8A — Generate `submission.csv` (GPU inference, not training)

In [14]:
SUBMISSION=OUT/'submission.csv'
SUBMISSION_META=OUT/'submission_settings.json'
if SUBMISSION.exists():
    raise RuntimeError('submission.csv already exists. Rename/delete it deliberately before generating a new version.')

model=YOLO(str(SUBMIT_CHECKPOINT))
rows=[]
B=4
for start in range(0,len(wanted_ids),B):
    ids=wanted_ids[start:start+B]
    files=[str(image_dict[i]) for i in ids]
    results=model.predict(source=files, imgsz=SUBMIT_IMGSZ,device=0,batch=B,
                          conf=SUBMIT_CONF,iou=SUBMIT_IOU,max_det=SUBMIT_MAX_DET,
                          augment=False,verbose=False,save=False)
    assert len(results)==len(ids)
    for image_id,r in zip(ids,results):
        det=[]
        if r.boxes is not None:
            for box,cls,score in zip(r.boxes.xyxy.cpu().numpy(),r.boxes.cls.cpu().numpy(),r.boxes.conf.cpu().numpy()):
                x1,y1,x2,y2=map(float,box)
                w,h=max(0.,x2-x1),max(0.,y2-y1)
                if w<=0 or h<=0:continue
                cls=int(cls)
                assert 0<=cls<8
                det.extend([str(cls),f'{float(score):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}'])
        rows.append({'image_id':image_id,'PredictionString':' '.join(det)})
    if (start//B+1)%50==0:
        print('Test predictions:',min(start+B,len(wanted_ids)),'/',len(wanted_ids))

submission=pd.DataFrame(rows)
assert len(submission)==len(wanted_ids)
assert submission['image_id'].tolist()==wanted_ids
assert not submission['image_id'].duplicated().any()
assert submission['PredictionString'].notna().all()
# Check every detection is six tokens: cls score x y w h.
for s in submission['PredictionString']:
    t=s.split()
    assert len(t)%6==0
    if t:
        arr=np.array(t,dtype=float).reshape(-1,6)
        assert np.isfinite(arr).all()
        assert np.all((arr[:,0]>=0)&(arr[:,0]<8)&(np.floor(arr[:,0])==arr[:,0]))
        assert np.all((arr[:,1]>=0)&(arr[:,1]<=1))
        assert np.all(arr[:,4:6]>0)
submission.to_csv(SUBMISSION,index=False)
SUBMISSION_META.write_text(json.dumps({
    'checkpoint':str(SUBMIT_CHECKPOINT),'imgsz':SUBMIT_IMGSZ,
    'conf':SUBMIT_CONF,'iou':SUBMIT_IOU,'max_det':SUBMIT_MAX_DET,
    'n_test_images':len(wanted_ids),'format':'class confidence x_min y_min width height, YOLO 0-based classes'
},indent=2))
print('SUBMISSION WRITTEN:',SUBMISSION,'size MB',round(SUBMISSION.stat().st_size/1e6,1))
display(submission[['image_id']].head())
del model;gc.collect();torch.cuda.empty_cache()

Test predictions: 200 / 3811
Test predictions: 400 / 3811
Test predictions: 600 / 3811
Test predictions: 800 / 3811
Test predictions: 1000 / 3811
Test predictions: 1200 / 3811
Test predictions: 1400 / 3811
Test predictions: 1600 / 3811
Test predictions: 1800 / 3811
Test predictions: 2000 / 3811
Test predictions: 2200 / 3811
Test predictions: 2400 / 3811
Test predictions: 2600 / 3811
Test predictions: 2800 / 3811
Test predictions: 3000 / 3811
Test predictions: 3200 / 3811
Test predictions: 3400 / 3811
Test predictions: 3600 / 3811
Test predictions: 3800 / 3811
SUBMISSION WRITTEN: /kaggle/working/yolo_leaderboard_v1/submission.csv size MB 18.2


,image_id
0,clip_001_0000001
1,clip_001_0000002
2,clip_001_0000003
3,clip_001_0000004
4,clip_001_0000005


## 9 — Save essentials and inspect outputs

Create a **portable** backup with `best.pt`, training history, settings and metrics; `last.pt` is deliberately omitted to avoid a redundant large archive. Keep the original unarchived `last.pt` if you intend to resume. For the Kaggle submission, `submission.csv` is also a standalone file under `/kaggle/working/yolo_leaderboard_v1/`.

**IMPORTANT:** ZIP inside `/kaggle/working` is still temporary. Download via Kaggle **Output** or save a notebook version with output files; Quick Save is not a reliable checkpoint backup.

In [15]:
BACKUP=Path('/kaggle/working/yolo_leaderboard_essentials.zip')
selected=[]
for p in OUT.rglob('*'):
    if not p.is_file():continue
    if p.name in ('best.pt','completed.json','holdout_coco_metrics.json','results.csv','args.yaml','submission_settings.json'):
        selected.append(p)
    elif p.parent==SPLITS and p.suffix in ('.csv','.yaml'):
        selected.append(p)
# Do not include submission.csv unless tiny? Keep it standalone in Output.
with zipfile.ZipFile(BACKUP,'w',compression=zipfile.ZIP_STORED) as z:
    for p in selected:z.write(p,arcname=str(p.relative_to(OUT)))
with zipfile.ZipFile(BACKUP) as z:assert z.testzip() is None
print('Backup:',BACKUP)
print('Size MB:',round(BACKUP.stat().st_size/1e6,1))
print('Files:',len(selected))
print('Submission:',OUT/'submission.csv','exists:',(OUT/'submission.csv').exists())
print('Remember to export checkpoints outside current draft session.')

Backup: /kaggle/working/yolo_leaderboard_essentials.zip
Size MB: 19.9
Files: 9
Submission: /kaggle/working/yolo_leaderboard_v1/submission.csv exists: True
Remember to export checkpoints outside current draft session.
